In [0]:
CREATE LIVE TABLE diagnostic_mapping
COMMENT "Bronze table for the diagnosis mapping file"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT *
FROM incremental_load.default.raw_diagnosis_map

In [0]:
CREATE OR REFRESH STREAMING TABLE daily_patients
COMMENT "Bronze table for daily patient data"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT *
FROM STREAM(incremental_load.default.raw_patients_daily)

In [0]:
CREATE OR REFRESH STREAMING TABLE processed_patient_data(CONSTRAINT valid_data EXPECT (patient_id IS NOT NULL and `name` IS NOT NULL and age IS NOT NULL and gender IS NOT NULL and `address` IS NOT NULL and contact_number IS NOT NULL and admission_date IS NOT NULL) ON VIOLATION DROP ROW)
COMMENT "Silver table with newly joined data from bronze tables and data quality constraints"
TBLPROPERTIES ("quality" = "silver")
AS 
SELECT
    p.patient_id,
    p.name,
    p.age,
    p.gender,
    p.address,
    p.contact_number,
    p.admission_date,
    m.diagnosis_description
FROM STREAM(live.daily_patients) p
LEFT JOIN live.diagnostic_mapping m
ON p.diagnosis_code = m.diagnosis_code;

In [0]:
CREATE LIVE TABLE patients_statistics_by_description
COMMENT "Gold table with detailed patient statistical data based on diagnosis description"
TBLPROPERTIES ("quality" = 'gold')
AS
SELECT
    diagnosis_description,
    COUNT(patient_id) as total_patients,
    CAST(AVG(age) AS INT) as average_age,
    MIN(age) as min_age,
    MAX(age) as max_age
FROM live.processed_patient_data
group by diagnosis_description

In [0]:
CREATE LIVE TABLE patients_statistics_by_gender
COMMENT "Gold table with detailed patient statistical data based on gender"
TBLPROPERTIES ("quality" = 'gold')
AS
SELECT
    gender,
    COUNT(patient_id) as total_patients,
    CAST(AVG(age) AS INT) as average_age,
    MIN(age) as min_age,
    MAX(age) as max_age
FROM live.processed_patient_data
group by gender